# PostgreSQL functions check

This notebook helps verify that functions were created in the target schema.

Checks included:
- function count in schema
- full function list
- function signatures
- function definitions (preview)
- required function existence checks

In [1]:
import os
from pathlib import Path

import psycopg
from psycopg.rows import dict_row


def load_env(path: Path) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


load_env(Path.cwd().parent / ".env")

DB_CONFIG = {
    "host": os.getenv("PGHOST", "localhost"),
    "port": int(os.getenv("PGPORT", "5432")),
    "dbname": os.getenv("PGDATABASE"),
    "user": os.getenv("PGUSER"),
    "password": os.getenv("PGPASSWORD"),
}
SCHEMA = "s_grnplm_as_t_didsd_nnn_db_tmd"

conn = psycopg.connect(**DB_CONFIG, row_factory=dict_row)
print("Connected")

Connected


In [2]:
with conn.cursor() as cur:
    cur.execute(
        """
        SELECT count(*) AS function_count
        FROM pg_proc p
        JOIN pg_namespace n ON n.oid = p.pronamespace
        WHERE n.nspname = %s
        """,
        (SCHEMA,),
    )
    function_count = cur.fetchone()["function_count"]

print({"schema": SCHEMA, "function_count": function_count})

{'schema': 's_grnplm_as_t_didsd_nnn_db_tmd', 'function_count': 38}


In [3]:
# List function names
with conn.cursor() as cur:
    cur.execute(
        """
        SELECT proname AS function_name
        FROM pg_proc p
        JOIN pg_namespace n ON n.oid = p.pronamespace
        WHERE n.nspname = %s
        ORDER BY proname
        """,
        (SCHEMA,),
    )
    function_names = [r["function_name"] for r in cur.fetchall()]

function_names

['add_arr_log',
 'fn_create_obj_test1',
 'fn_dbc_partition_table_name_for',
 'fn_drop_gen_data_tables',
 'fn_explain_to_debug',
 'fn_gen_data_test1',
 'fn_gen_data_test2',
 'fn_gen_data_test3',
 'fn_generate_coa_data',
 'fn_generate_je_line_data',
 'fn_run_calc_d_agr_cred_coa_period_calc_dt_test3',
 'fn_run_calc_fltr_skew_test1',
 'fn_run_calc_fltr_test1',
 'fn_run_calc_main_debts_test2',
 'fn_run_calc_skew_test1',
 'fn_run_test1',
 'fn_run_test2',
 'fn_run_test3',
 'gen_random_uuid',
 'generate_agr_cred',
 'generate_agr_cred_coa',
 'generate_agr_cred_coa_period_prep_bal',
 'generate_agr_cred_optn',
 'generate_tech_tbl_coa_bal_h',
 'prc_debug',
 'prc_error_message',
 'save_prc_log',
 'sp_collect_stat',
 'sp_dm_ent_ld_d_agr_cred_coa_period_calc_dt',
 'sp_dm_ent_ld_d_agr_cred_coa_period_calc_dt_separate',
 'sp_dm_ent_ld_d_agr_cred_coa_period_prep_main_debts',
 'sp_dm_ent_ld_d_agr_cred_coa_period_prep_main_debts_separate',
 'sp_execute_run',
 'sp_get_calc_params',
 'sp_spt_je_header_fltr'

In [4]:
# Detailed signatures
with conn.cursor() as cur:
    cur.execute(
        """
        SELECT
            p.proname AS function_name,
            pg_get_function_identity_arguments(p.oid) AS args,
            pg_get_function_result(p.oid) AS returns
        FROM pg_proc p
        JOIN pg_namespace n ON n.oid = p.pronamespace
        WHERE n.nspname = %s
        ORDER BY p.proname
        """,
        (SCHEMA,),
    )
    signatures = cur.fetchall()

signatures

[{'function_name': 'add_arr_log',
  'args': 'in_workflow2_run_id bigint, strobject text, param text, sql_query text, INOUT vlog s_grnplm_as_t_didsd_nnn_db_tmd.tp_dm_log[]',
  'returns': 's_grnplm_as_t_didsd_nnn_db_tmd.tp_dm_log[]'},
 {'function_name': 'fn_create_obj_test1', 'args': '', 'returns': 'void'},
 {'function_name': 'fn_dbc_partition_table_name_for',
  'args': 'p_schema_name text, p_table_name text, p_partition_value text',
  'returns': 'text'},
 {'function_name': 'fn_drop_gen_data_tables', 'args': '', 'returns': 'void'},
 {'function_name': 'fn_explain_to_debug',
  'args': 'in_workflow_run_id bigint, p_sql text, p_in_param text, p_in_proc text, INOUT in_log s_grnplm_as_t_didsd_nnn_db_tmd.tp_dm_log[]',
  'returns': 's_grnplm_as_t_didsd_nnn_db_tmd.tp_dm_log[]'},
 {'function_name': 'fn_gen_data_test1',
  'args': 'in_coa_cnt bigint, in_je_line_cnt bigint, OUT return_int_return_code integer',
  'returns': 'integer'},
 {'function_name': 'fn_gen_data_test2',
  'args': 'in_num_records 

In [5]:
# Definition preview for one function name (change as needed)
FUNCTION_TO_INSPECT = "fn_gen_data_test1"

with conn.cursor() as cur:
    cur.execute(
        """
        SELECT
            p.proname,
            pg_get_functiondef(p.oid) AS definition
        FROM pg_proc p
        JOIN pg_namespace n ON n.oid = p.pronamespace
        WHERE n.nspname = %s
          AND p.proname = %s
        ORDER BY p.oid
        """,
        (SCHEMA, FUNCTION_TO_INSPECT),
    )
    defs = cur.fetchall()

defs

[{'proname': 'fn_gen_data_test1',
  'definition': "CREATE OR REPLACE FUNCTION s_grnplm_as_t_didsd_nnn_db_tmd.fn_gen_data_test1(in_coa_cnt bigint, in_je_line_cnt bigint, OUT return_int_return_code integer)\n RETURNS integer\n LANGUAGE plpgsql\nAS $function$\nDECLARE\n   v_start_time TIMESTAMP := clock_timestamp();\n   _log text;\n   vlog s_grnplm_as_t_didsd_nnn_db_tmd.tp_dm_log array;\n   v_workflow_run_id bigint:= (select abs(uuid_hash(s_grnplm_as_t_didsd_nnn_db_tmd.gen_random_uuid()))::bigint);\n   v_proc text :='fn_run_test1_'||in_je_line_cnt::text;\n   v_error_msg text;\n   v_sql text;\n   v_full_error_msg text;\n   v_partition_table_full_name text;\n   tmp_result_code int default 0;\nBEGIN\nbegin\n -- очищаем и заполняем систетику\ndelete from s_grnplm_as_t_didsd_nnn_db_tmd.t_str_pilot_tune where test_num='test1';\n\n  return_int_return_code = 0;\n  tmp_result_code = 1;\n  vlog = s_grnplm_as_t_didsd_nnn_db_tmd.add_arr_log(v_workflow_run_id, v_proc, 'START', 'workflow_run_id='||v_wo

In [6]:
# Check required functions exist
required_functions = [
    "fn_create_obj_test1",
    "fn_gen_data_test1",
    "fn_gen_data_test2",
    "fn_gen_data_test3",
    "fn_run_test1",
    "fn_run_test2",
    "fn_run_test3",
    "fn_drop_gen_data_tables",
    "fn_dbc_partition_table_name_for",
]

with conn.cursor() as cur:
    cur.execute(
        """
        SELECT proname
        FROM pg_proc p
        JOIN pg_namespace n ON n.oid = p.pronamespace
        WHERE n.nspname = %s
        """,
        (SCHEMA,),
    )
    existing = {r["proname"] for r in cur.fetchall()}

check_result = [
    {
        "function": fn,
        "exists": fn in existing,
    }
    for fn in required_functions
]

check_result

[{'function': 'fn_create_obj_test1', 'exists': True},
 {'function': 'fn_gen_data_test1', 'exists': True},
 {'function': 'fn_gen_data_test2', 'exists': True},
 {'function': 'fn_gen_data_test3', 'exists': True},
 {'function': 'fn_run_test1', 'exists': True},
 {'function': 'fn_run_test2', 'exists': True},
 {'function': 'fn_run_test3', 'exists': True},
 {'function': 'fn_drop_gen_data_tables', 'exists': True},
 {'function': 'fn_dbc_partition_table_name_for', 'exists': True}]

In [7]:
conn.close()
print("Connection closed")

Connection closed
